# 04 — Frozen Lake + Exploration
**Week 3 | RL Fundamentals**

First contact with **Gymnasium**. We run various policies on Frozen Lake and measure performance — no learning yet, just understanding the environment and the exploration-exploitation trade-off.

In [ ]:
# Install gymnasium if needed
try:
    import gymnasium as gym
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gymnasium', '-q'])
    import gymnasium as gym

import numpy as np
import matplotlib.pyplot as plt
print('Gymnasium version:', gym.__version__)

In [ ]:
env = gym.make('FrozenLake-v1', is_slippery=False)
print(f"Observation space: {env.observation_space}  ({env.observation_space.n} states)")
print(f"Action space:      {env.action_space}  (0=L, 1=D, 2=R, 3=U)")
print(f"\nTransition model for state 0, action 1 (DOWN):")
for prob, next_s, reward, done in env.unwrapped.P[0][1]:
    print(f"  P={prob:.2f} -> state {next_s}, reward={reward}, done={done}")

## 1. Random Policy Baseline

In [ ]:
def evaluate_policy(env, policy_fn, n_episodes=1000, max_steps=200):
    wins = 0
    returns = []
    for _ in range(n_episodes):
        s, _ = env.reset()
        ep_return = 0
        for _ in range(max_steps):
            a = policy_fn(s)
            s, r, terminated, truncated, _ = env.step(a)
            ep_return += r
            if terminated or truncated:
                if r == 1.0: wins += 1
                break
        returns.append(ep_return)
    return wins / n_episodes, np.mean(returns)

random_win, random_ret = evaluate_policy(env, lambda s: env.action_space.sample())
print(f"Random policy:  win rate={random_win:.2%}, avg return={random_ret:.4f}")

## 2. Fixed Directional Policy
Manually specify the best deterministic path.

In [ ]:
# Hand-crafted policy for 4x4 non-slippery FrozenLake (actions: 0=L,1=D,2=R,3=U)
# Map: SFFF / FHFH / FFFH / HFFG
hand_policy = [
    1, 2, 1, 0,   # row 0
    1, 0, 1, 0,   # row 1 (H at pos 5,7)
    2, 2, 1, 0,   # row 2
    0, 2, 2, 0,   # row 3 (H at pos 12, G at 15)
]
hand_win, hand_ret = evaluate_policy(env, lambda s: hand_policy[s])
print(f"Hand policy:    win rate={hand_win:.2%}, avg return={hand_ret:.4f}")

## 3. ε-greedy Policy with Learned Bias
Start with a 'smart' action bias and add exploration noise.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
eps_values = np.linspace(0, 1, 20)
results = []
for eps in eps_values:
    def eps_hand(s, eps=eps):
        if np.random.rand() < eps:
            return env.action_space.sample()
        return hand_policy[s]
    w, _ = evaluate_policy(env, eps_hand, n_episodes=500)
    results.append(w)
ax.plot(eps_values, results, color='steelblue', linewidth=2, marker='o', markersize=5)
ax.axhline(random_win, color='tomato', linestyle='--', label=f'Random baseline ({random_win:.0%})')
ax.set_xlabel('ε (exploration rate)'); ax.set_ylabel('Win rate')
ax.set_title('Trade-off: Exploration vs Exploitation on FrozenLake')
ax.legend(); plt.tight_layout(); plt.show()

## 4. Inspect the MDP Transitions

In [ ]:
# Show how slippery=True changes things
env_slip = gym.make('FrozenLake-v1', is_slippery=True)
slip_win, _ = evaluate_policy(env_slip, lambda s: hand_policy[s])
print(f"Hand policy on SLIPPERY lake: win rate={slip_win:.2%}")
print("\n(Our deterministic policy fails on a stochastic environment!)")

## ✅ Exercises
1. Try the 8×8 FrozenLake map (`map_name='8x8'`). Does the hand-crafted policy still work? Why not?
2. Plot win rate vs number of evaluation episodes for the random policy. How many episodes do you need for a stable estimate?
3. **Challenge**: write a systematic policy for the slippery lake — one that avoids dangerous edges. Test it.

## Q1: 8×8 FrozenLake — Does the hand-crafted policy still work?
No. The hand-crafted policy was manually defined for the 4×4 map (16 states). The 8×8 map has 64 states with a completely different hole layout, so the policy is both incomplete (no entries for states 16–63) and incorrect for the new grid structure.

## Q2: Win Rate vs Number of Evaluation Episodes

In [ ]:
episode_counts = [10, 50, 100, 200, 500, 1000, 2000, 5000]
win_rates = []
for n in episode_counts:
    w, _ = evaluate_policy(env, lambda s: env.action_space.sample(), n_episodes=n)
    win_rates.append(w)

plt.figure(figsize=(8, 3.5))
plt.plot(episode_counts, win_rates, color='steelblue', linewidth=2, marker='o', markersize=5)
plt.axhline(random_win, color='tomato', linestyle='--', label=f'True baseline ({random_win:.2%})')
plt.xscale('log')
plt.xlabel('Number of evaluation episodes (log scale)')
plt.ylabel('Estimated win rate')
plt.title('Stability of Win Rate Estimate vs Episode Count')
plt.legend(); plt.tight_layout(); plt.show()

The estimate stabilizes around 500–1000 episodes. Below 100 episodes the variance is too high to trust the estimate.

## Q3: Systematic Policy for Slippery Lake

The strategy is to consistently move down and right toward the goal, avoiding cells adjacent to holes, particularly along the left column and top row where slipping can push the agent into holes.

In [ ]:
systematic_policy = [
    2, 2, 1, 0,
    1, 1, 1, 1,
    2, 2, 2, 1,
    3, 2, 2, 0,
]

env_slip = gym.make('FrozenLake-v1', is_slippery=True)
slip_win_random,     _ = evaluate_policy(env_slip, lambda s: env_slip.action_space.sample())
slip_win_systematic, _ = evaluate_policy(env_slip, lambda s: systematic_policy[s])

print(f"Random policy     win rate: {slip_win_random:.2%}")
print(f"Systematic policy win rate: {slip_win_systematic:.2%}")

The systematic policy outperforms the random policy on the slippery lake. However, since transitions are stochastic, no deterministic policy can guarantee a high win rate — the improvement is probabilistic rather than certain.